In [2]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device.type

'cuda'

In [4]:
%load_ext autoreload
%autoreload 2

from drGAT import drGAT

In [5]:
train_data = pd.read_csv("../CTRP_data/train.csv")
val_data = pd.read_csv("../CTRP_data/val.csv")
test_data = pd.read_csv("../CTRP_data/test.csv")

In [6]:
train_data.head()

,Drug,Cell
0,YM-155,SKM-1
1,austocystin D,RCC10RGB
2,sorafenib,SK-HEP-1
3,UNC0638,RT4
4,cabozantinib,SNU-5


In [7]:
idxs = np.load("../CTRP_data/idxs.npy", allow_pickle=True)
converter = {idxs[1, i]: int(idxs[0, i]) for i in range(idxs.shape[1])}

In [8]:
edge_index = np.load("../CTRP_data/edge_idxs.npy")
edge_attr = np.load("../CTRP_data/edge_attr.npy")

In [9]:
def get_idx(X):
    X["Drug"] = [converter[(i)] for i in X["Drug"]]
    X["Cell"] = [converter[(i)] for i in X["Cell"]]
    return X

In [10]:
train_data = get_idx(train_data)
val_data = get_idx(val_data)
test_data = get_idx(test_data)

In [11]:
edge_index = torch.tensor(edge_index).int()
edge_index = edge_index.type(torch.int64)

In [12]:
edge_attr = torch.tensor(edge_attr).float()

In [13]:
train_drug = train_data["Drug"].values
train_cell = train_data["Cell"].values
val_drug = val_data["Drug"].values
val_cell = val_data["Cell"].values

In [14]:
train_labels = np.load("../CTRP_data/train_labels.npy")
val_labels = np.load("../CTRP_data/val_labels.npy")

train_labels = torch.tensor(train_labels).float()
val_labels = torch.tensor(val_labels).float()

## Get feature matrix

In [15]:
drug = pd.read_csv("../CTRP_data/drug_sim.csv", index_col=0)
cell = pd.read_csv("../CTRP_data/cell_sim.csv", index_col=0)
gene = pd.read_csv("../CTRP_data/gene_sim.csv", index_col=0)

In [16]:
drug = torch.tensor(drug.values).float()
cell = torch.tensor(cell.values).float()
gene = torch.tensor(gene.values).float()

# Create the dataset

In [17]:
data = [
    drug,
    cell,
    gene,
    edge_index,
    edge_attr,
    train_drug,
    train_cell,
    val_drug,
    val_cell,
    train_labels,
    val_labels,
]

In [18]:
params = {
    "dropout1": 0.1,
    "dropout2": 0.1,
    "n_drug": drug.shape[0],
    "n_cell": cell.shape[0],
    "n_gene": gene.shape[0],
    "hidden1": 256,
    "hidden2": 32,
    "hidden3": 128,
    "epochs": 3,
    "lr": 0.001,
    "heads": 2,
}

In [19]:
def objective(trial):
    params = {
        "n_drug": drug.shape[0],
        "n_cell": cell.shape[0],
        "n_gene": gene.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        "epochs": trial.suggest_int("epochs", 10, 200, step=50),
        "heads": trial.suggest_categorical("heads", [1, 2, 3]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer", ["GAT", "GATv2", "Transformer"]
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        _, _, _, best_metrics, early_stopping_epoch = drGAT.train(
            data, params=params, device=device, verbose=False
        )
        print("#####")
        print(best_metrics)
        print("#####")

        early_stop_threshold = trial.suggest_float("early_stop_threshold", 0.3, 0.7)
        if (
            early_stopping_epoch is not None
            and early_stopping_epoch < params["epochs"] * early_stop_threshold
        ):
            raise optuna.TrialPruned("Early stopping occurred too early")

        trial.set_user_attr("early_stopping_epoch", early_stopping_epoch)
        return best_metrics

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print("CUDA out of memory")
            trial.set_user_attr("status", "CUDA OOM")

            torch.cuda.empty_cache()
            gc.collect()

            return [None] * 4
        else:
            raise e

In [ ]:
name = "CTRP_GAT"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}.sqlite3".format(name),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100)

[I 2025-03-21 17:01:55,450] Using an existing study with name 'CTRP_GAT' instead of creating a new one.
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/distributions.py:700: UserWarning: The distribution is specified by [10, 200] and step=50, but the range is not divisible by `step`. It will be replaced by [10, 160].
  warnings.warn(
[I 2025-03-21 17:01:55,908] Trial 2 pruned. Memory intensive configuration


Using:  cuda


100%|██████████| 60/60 [00:35<00:00,  1.69it/s]
[I 2025-03-21 17:02:32,191] Trial 3 finished with values: [0.829235960472403, 0.9014129679300726, 0.9025691346348286, 0.8286785152944022] and parameters: {'dropout1': 0.2, 'dropout2': 0.1, 'hidden1': 256, 'hidden2': 128, 'hidden3': 128, 'epochs': 60, 'heads': 2, 'activation': 'swish', 'optimizer': 'SGD', 'lr': 0.0016482007761046023, 'weight_decay': 0.0009291565621845429, 'scheduler': 'Step', 'gnn_layer': 'GATv2', 'gamma_step': 0.84042410266579, 'step_size': 10, 'momentum': 0.8002815301349936, 'nesterov': False, 'early_stop_threshold': 0.604442057849736}. 


Best model found at epoch 60
#####
[0.829235960472403, 0.9014129679300726, 0.9025691346348286, 0.8286785152944022]
#####


/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/distributions.py:700: UserWarning: The distribution is specified by [10, 200] and step=50, but the range is not divisible by `step`. It will be replaced by [10, 160].
  warnings.warn(
[I 2025-03-21 17:02:32,545] Trial 4 pruned. Invalid layer size configuration


...

100%|██████████| 160/160 [02:43<00:00,  1.02s/it]


Best model found at epoch 160
#####
[0.8228488792480115, 0.8995437613585888, 0.904819137408701, 0.8175273088381331]
#####


[I 2025-03-21 17:22:22,997] Trial 250 finished with values: [0.8228488792480115, 0.8995437613585888, 0.904819137408701, 0.8175273088381331] and parameters: {'dropout1': 0.1, 'dropout2': 0.3, 'hidden1': 1028, 'hidden2': 256, 'hidden3': 64, 'epochs': 160, 'heads': 3, 'activation': 'relu', 'optimizer': 'SGD', 'lr': 0.006112824099968585, 'weight_decay': 0.0005053949484624101, 'scheduler': None, 'gnn_layer': 'Transformer', 'momentum': 0.8206522679796009, 'nesterov': True, 'early_stop_threshold': 0.5466601332755264}. 
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/distributions.py:700: UserWarning: The distribution is specified by [10, 200] and step=50, but the range is not divisible by `step`. It will be replaced by [10, 160].
  warnings.warn(


Using:  cuda


 75%|███████▍  | 82/110 [01:22<00:28,  1.01s/it]

## Eval

In [ ]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [22]:
# prob, res, test_attention = drGAT.eval(model, test)